In [ ]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import mlflow
import optuna
import datetime

from sklearn.model_selection import (
    train_test_split,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, f1_score

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Bank-Customer-Churn-Prediction-Experiment")

# Data preprocessing

In [ ]:
data_path = "../data/Customer-Churn-Records.csv"


def clean_data(data_path):
    df = pd.read_csv(data_path)
    df.head()

    return df


def split_data(df):
    # Train/val/test stratified split of ratio 0.8/0.1/0.1
    labels = df.Exited.values
    del df["Exited"]

    X_train, X_vtest, y_train, y_vtest = train_test_split(
        df, labels, test_size=0.2, random_state=seed, stratify=labels
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_vtest, y_vtest, test_size=0.5, random_state=seed, stratify=y_vtest
    )

    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
cat = [
    "Geography",
    "Gender",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "Satisfaction Score",
    "Card Type",
]

num = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary", "Point Earned"]


def preprocess_data(X_train, X_val, X_test):
    preprocessor = ColumnTransformer(
        [
            ("oh", OneHotEncoder(handle_unknown="ignore"), cat),
            ("scaler", StandardScaler(), num),
        ]
    )

    X_train = preprocessor.fit_transform(X_train)
    X_val = preprocessor.transform(X_val)
    X_test = preprocessor.transform(X_test)

    with mlflow.start_run():
        mlflow.sklearn.log_model(preprocessor, "ChurnDataPreprocessor")

    return X_train, X_val, X_test, preprocessor

In [ ]:
records = clean_data(data_path)
X_train, y_train, X_val, y_val, X_test, y_test = split_data(records)
X_train_tf, X_val_tf, X_test_tf, pp = preprocess_data(X_train, X_val, X_test)

# Model Evaluation and Hyperparameters Tuning

In [ ]:
sampler = optuna.samplers.TPESampler(seed=seed)

In [ ]:
def xgb_objective(trial):
    with mlflow.start_run(nested=True):
        train = xgb.DMatrix(X_train_tf, label=y_train)
        valid = xgb.DMatrix(X_val_tf, label=y_val)

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 5000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-9, 100.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-9, 100.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.1, 1.0),
            "max_depth": trial.suggest_int("max_depth", 1, 12),
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 1e-9, 0.5, log=True),
            "scale_pos_weight": trial.suggest_float(
                "scale_pos_weight", 1e-6, 500.0, log=True
            ),
            "seed": seed,
        }

        model = xgb.train(
            params,
            train,
            evals=[(valid, "validation")],
            early_stopping_rounds=300,
            verbose_eval=False,
        )

        preds = model.predict(valid)
        pred_labels = np.clip(np.rint(preds), 0, 1)

        f1 = f1_score(y_val, pred_labels)
        roc_auc = roc_auc_score(y_val, pred_labels)

        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_params(params)

    return roc_auc


def train_best_xgb_model(X_train, y_train, best_params):
    train = xgb.DMatrix(X_train, label=y_train)

    model = xgb.train(best_params, train)

    return model


def plot_feature_importance(model, feat_names=None):
    """
    Plots feature importance for an XGBoost model.

    Args:
    - model: A trained XGBoost model

    Returns:
    - fig: The matplotlib figure object
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    importance_type = "gain"
    if feat_names is not None:
        model.feature_names = list(feat_names)

    xgb.plot_importance(
        model,
        importance_type=importance_type,
        ax=ax,
        title=f"Feature Importance based on {importance_type}",
    )
    plt.tight_layout()
    plt.close(fig)

    return fig


def hyperparameter_tuning(X_train, y_train, feat_names=None):
    with mlflow.start_run(
        run_name=f"xgboost_hyperparameter_tuning_{datetime.datetime.now().date()}",
        nested=True,
    ):
        mlflow.set_tag("model", "xgboost")
        study_xgb = optuna.create_study(direction="maximize", sampler=sampler)
        study_xgb.optimize(xgb_objective, n_trials=200)

        print("Number of finished trials:", len(study_xgb.trials))
        print("Best value:", study_xgb.best_value)

        mlflow.log_params(study_xgb.best_params)

        xgb_model = train_best_xgb_model(X_train, y_train, study_xgb.best_params)

        mlflow.xgboost.log_model(
            xgb_model=xgb_model,
            name="mlflow_model",
            input_example=X_train[:5],
            registered_model_name="XGBoostChurnModel",
        )

        importances = plot_feature_importance(
            xgb_model,
            feat_names=feat_names,
        )
        mlflow.log_figure(figure=importances, artifact_file="feature_importances.png")

In [ ]:
hyperparameter_tuning(
    X_train=X_train_tf, y_train=y_train, feat_names=pp.get_feature_names_out()
)

In [ ]:
loaded_preprocessor = mlflow.sklearn.load_model("models:/ChurnDataPreprocessor/latest")
loaded_model = mlflow.xgboost.load_model("models:/XGBoostChurnModel/latest")

In [ ]:
def make_predictions(
    model, X_test: pd.DataFrame, preprocessor: ColumnTransformer
) -> np.ndarray:
    X_test = preprocessor.transform(X_test)
    preds = model.predict(xgb.DMatrix(X_test))
    return np.clip(np.rint(preds), 0, 1).astype(int)

In [ ]:
preds = make_predictions(loaded_model, X_test, loaded_preprocessor)

In [ ]:
preds_tr = make_predictions(loaded_model, X_train, loaded_preprocessor)
X_train = X_train.assign(Preds=preds_tr)
preds_val = make_predictions(loaded_model, X_val, loaded_preprocessor)
X_val = X_val.assign(Preds=preds_val)
preds_t = make_predictions(loaded_model, X_test, loaded_preprocessor)
X_test = X_test.assign(Preds=preds_t)

In [ ]:
X_trv = pd.concat([X_train, X_val], axis=0)
X_trv

# Evidently Report

In [ ]:
from evidently import Report, DataDefinition, Dataset
from evidently.presets import DataDriftPreset
from evidently.ui.workspace import RemoteWorkspace
from evidently.metrics import ValueDrift, DriftedColumnsCount, MissingValueCount

ws = RemoteWorkspace("http://localhost:8000")

In [ ]:
if proj_list := ws.search_project("Churn Prediction Project"):
    proj_id = proj_list[0].id
    project = ws.get_project(proj_id)
else:
    project = ws.create_project(name="Churn Prediction Project")

In [ ]:
data_definition = DataDefinition(
    numerical_columns=num,
    categorical_columns=cat + ["Exited", "Preds"],
)

cur_data = Dataset.from_pandas(
    data=X_train.assign(Exited=y_train),
    data_definition=data_definition,
)

ref_data = Dataset.from_pandas(
    data=X_test.assign(Exited=y_test),
    data_definition=data_definition,
)

report = Report(
    [
        ValueDrift(column="Preds"),
        DriftedColumnsCount(),
        MissingValueCount(column="Preds"),
        DataDriftPreset(),
    ],
    include_tests=True,
)

eval = report.run(cur_data, reference_data=ref_data)

In [ ]:
ws.add_run(project.id, eval)

In [ ]:
eval.dict()["metrics"]

In [ ]:
results = eval.dict()
prediction_drift = results["metrics"][0]["value"]
num_drifted_columns = results["metrics"][1]["value"]["count"]
share_missing_values = results["metrics"][2]["value"]["share"]

In [ ]:
CONNECTION_STRING = "host=localhost port=5434 user=grafana password=grafana"
CONNECTION_STRING_DB = CONNECTION_STRING + " dbname=grafana"

create_table_statement = """
drop table if exists metrics;
create table metrics(
	timestamp timestamp,
	prediction_drift float,
	num_drifted_columns integer,
	share_missing_values float
)
"""

In [ ]:
import psycopg

with psycopg.connect(CONNECTION_STRING, autocommit=True) as conn:
    res = conn.execute("SELECT 1 FROM pg_database WHERE datname='grafana'")
    if len(res.fetchall()) == 0:
        conn.execute("create database grafana;")
    with psycopg.connect(CONNECTION_STRING_DB) as conn:
        conn.execute(create_table_statement)

In [ ]:
with psycopg.connect(CONNECTION_STRING_DB, autocommit=True) as conn:
    with conn.cursor() as curr:
        curr.execute(
            "insert into metrics(timestamp, prediction_drift, num_drifted_columns, share_missing_values) values (%s, %s, %s, %s)",
            (
                datetime.date.today(),
                prediction_drift,
                num_drifted_columns,
                share_missing_values,
            ),
        )